In [2]:
# Import the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [3]:
# Check your dataset location
print(os.getcwd())
print(os.listdir())

/content
['.config', 'properties.csv', 'clients.csv', 'sample_data']


In [4]:
#Load the datasets
clients = pd.read_csv("clients.csv")
properties = pd.read_csv("properties.csv")

print("Clients dataset:", clients.shape)
print("Properties dataset:", properties.shape)

display(clients.head())
display(properties.head())

Clients dataset: (2000, 12)
Properties dataset: (10000, 9)


,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,Yes,Website
1,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,No,Website
2,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,Yes,Agency
3,C0004,Individual,Russell,Gross,11/25/1959,M,USA,California,Home,5,No,Website
4,C0005,Company,Marleez,Co,2/28/1976,M,USA,California,Investment,5,No,Website


,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,1012,1,01-01-2024,Apartment,12,1160.36,"$300,385.62",Sold,C0027
1,1015,1,01-01-2024,Apartment,15,782.25,"$208,930.81",Sold,C0097
2,1021,1,01-01-2024,Apartment,21,756.21,"$218,585.92",Sold,C0113
3,1030,1,01-01-2024,Apartment,30,743.09,"$246,172.68",Sold,C0141
4,2016,2,01-01-2024,Apartment,16,701.66,"$212,265.67",Sold,C0146


In [5]:
# Inspect the data structure
print("CLIENTS INFORMATION")
clients.info()

print("\nPROPERTIES INFORMATION")
properties.info()

CLIENTS INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   client_id            2000 non-null   object
 1   client_type          2000 non-null   object
 2   first_name           2000 non-null   object
 3   last_name            2000 non-null   object
 4   date_of_birth        2000 non-null   object
 5   gender               2000 non-null   object
 6   country              2000 non-null   object
 7   region               2000 non-null   object
 8   acquisition_purpose  2000 non-null   object
 9   satisfaction_score   2000 non-null   int64 
 10  loan_applied         2000 non-null   object
 11  referral_channel     2000 non-null   object
dtypes: int64(1), object(11)
memory usage: 187.6+ KB

PROPERTIES INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 # 

In [6]:
# Check missing values
print("Missing values in clients:")
display(clients.isnull().sum())

print("\nMissing values in properties:")
display(properties.isnull().sum())

Missing values in clients:


,0
client_id,0
client_type,0
first_name,0
last_name,0
date_of_birth,0
gender,0
country,0
region,0
acquisition_purpose,0
satisfaction_score,0



Missing values in properties:


,0
listing_id,0
tower_number,0
transaction_date,0
unit_category,0
unit_number,0
floor_area_sqft,0
sale_price,0
listing_status,0
client_ref,2695


In [7]:
# Check duplicates
print("Duplicate client rows:", clients.duplicated().sum())
print("Duplicate property rows:", properties.duplicated().sum())

print(
    "Duplicate client IDs:",
    clients["client_id"].duplicated().sum()
)

print(
    "Duplicate listing IDs:",
    properties["listing_id"].duplicated().sum()
)

Duplicate client rows: 0
Duplicate property rows: 0
Duplicate client IDs: 0
Duplicate listing IDs: 0


In [8]:
# Standardize column names and text
for df in [clients, properties]:
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )
        df[col] = df[col].replace("", np.nan)

print("Column names and text values standardized.")

Column names and text values standardized.


In [9]:
# Convert numeric values
clients["satisfaction_score"] = pd.to_numeric(
    clients["satisfaction_score"],
    errors="coerce"
)

properties["sale_price"] = pd.to_numeric(
    properties["sale_price"]
    .astype("string")
    .str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)

properties["floor_area_sqft"] = pd.to_numeric(
    properties["floor_area_sqft"],
    errors="coerce"
)

print("Numeric conversion completed.")

Numeric conversion completed.


In [10]:
# Validate dates
print(clients["date_of_birth"].head(10))
print(properties["transaction_date"].head(10))

0    05-11-1968
1    11/26/1962
2    04-07-1959
3    11/25/1959
4     2/28/1976
5    03-06-1957
6     5/24/1947
7    10/17/1969
8    10-05-1975
9     6/17/1966
Name: date_of_birth, dtype: object
0    01-01-2024
1    01-01-2024
2    01-01-2024
3    01-01-2024
4    01-01-2024
5    01-01-2024
6    01-01-2024
7    01-01-2024
8    01-01-2024
9    01-01-2024
Name: transaction_date, dtype: object


In [11]:
clients["date_of_birth"] = pd.to_datetime(
    clients["date_of_birth"],
    format="mixed",
    errors="coerce"
)

properties["transaction_date"] = pd.to_datetime(
    properties["transaction_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [12]:
# Validate the client-property relationship
valid_client_ids = set(clients["client_id"].dropna())

unmatched_refs = (
    properties["client_ref"].notna()
    & ~properties["client_ref"].isin(valid_client_ids)
)

print(
    "Missing client references:",
    properties["client_ref"].isna().sum()
)

print(
    "Unmatched client references:",
    unmatched_refs.sum()
)

Missing client references: 2695
Unmatched client references: 0


In [13]:
# Save the cleaned datasets
clients.to_csv("clients_cleaned.csv", index=False)
properties.to_csv("properties_cleaned.csv", index=False)

print("Both cleaned datasets saved successfully.")

Both cleaned datasets saved successfully.
